# 3 - Extração de Informações Estruturadas

**Extração** é a tarefa de identificar e isolar informações específicas dentro de um texto não estruturado
— como datas, nomes, eventos, valores — e retorná-las em um formato organizado (JSON/Pydantic).

É diferente do **Tagging** (notebook anterior): enquanto o tagging *classifica* o texto inteiro,
a extração *coleta* múltiplos dados estruturados que estão espalhados pelo conteúdo.

```
Texto livre  ──►  LLM + Schema  ──►  Objeto Pydantic estruturado

"A Microsoft foi fundada em 1975 por Bill Gates..."
        │
        ▼
EventsList(events=[
    Event(date='1975-04-04', event='Fundação da Microsoft...'),
    Event(date='1980',       event='Contrato com a IBM...'),
])
```

### O que mudou na API moderna

| Antigo (deprecado) | Moderno |
|--------------------|--------|
| `from langchain.pydantic_v1 import BaseModel` | `from pydantic import BaseModel` |
| `convert_to_openai_function(Schema)` | não necessário |
| `chat.bind(functions=[...], function_call={...})` | `chat.with_structured_output(Schema)` |
| `JsonOutputFunctionsParser()` | não necessário |
| `JsonKeyOutputFunctionsParser(key_name='posts')` | acesso direto via `resultado.posts` |

### Pacotes necessários

```bash
pip install langchain-openai langchain-community pydantic python-dotenv
```

In [ ]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

chat = ChatOpenAI(model="gpt-4o-mini")

## 1. Texto de exemplo

Usaremos um texto sobre a história da Microsoft para extrair eventos com datas.
O texto é não estruturado — as informações estão misturadas em parágrafos.

In [ ]:
texto = """
A Microsoft foi fundada em 4 de abril de 1975 por Bill Gates e Paul Allen, em Albuquerque, 
no estado do Novo México. O nome "Microsoft" é uma combinação das palavras "microcomputador" 
e "software", refletindo o foco da empresa em software para computadores pessoais.
O primeiro grande projeto da Microsoft foi a criação de um sistema operacional para o 
computador Altair 8800, um dos primeiros microcomputadores disponíveis comercialmente. 
O sistema, denominado Altair BASIC, foi desenvolvido em parceria com a MITS 
(Micro Instrumentation and Telemetry Systems) e foi um marco inicial para a Microsoft.
Em 1980, a empresa firmou um contrato significativo com a IBM para fornecer o sistema 
operacional para o novo PC da IBM, o que levou à criação do MS-DOS. Esse contrato foi um 
ponto de virada para a Microsoft, impulsionando sua expansão e dominando o mercado de 
sistemas operacionais para PCs nos anos seguintes.
Com o sucesso do MS-DOS, a Microsoft se consolidou como líder no setor de software e, 
em 1985, lançou o Windows, um sistema operacional gráfico que viria a se tornar a base 
de sua supremacia no mercado de sistemas operacionais para desktop.
"""
print(texto)

## 2. Definindo schemas para extração com Pydantic

Para extrair **listas** de objetos, definimos dois schemas encadeados:
- Um schema para o **item individual** (`Event`)
- Um schema para a **coleção** (`EventsList`) com uma lista do item

O `with_structured_output()` funciona com schemas aninhados e listas sem nenhuma configuração extra.

> **`List[Event]`**: o tipo `List` do `typing` instrui o modelo a retornar um array JSON
> com múltiplos objetos `Event`. O Pydantic valida cada item automaticamente.

In [ ]:
from pydantic import BaseModel, Field
from typing import List

class Event(BaseModel):
    """Informações sobre um evento ocorrido."""
    date: str  = Field(description="Data do evento no formato YYYY-MM-DD")
    event: str = Field(description="Descrição do evento extraído do texto")

class EventsList(BaseModel):
    """Lista de eventos extraídos do texto."""
    events: List[Event] = Field(
        description="Conjunto de eventos encontrados no texto fornecido"
    )

## 3. Chain de extração

A chain é idêntica ao padrão de tagging — a diferença está no schema:
aqui pedimos uma **lista de objetos** em vez de um objeto único.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "Extraia os acontecimentos e suas datas do texto fornecido."),
    ("human", "{input}")
])

# with_structured_output() substitui o antigo:
# chat.bind(functions=[tool_events], function_call={'name': 'EventsList'})
# | JsonOutputFunctionsParser()
chain = prompt | chat.with_structured_output(EventsList)

In [ ]:
# Executando a extração
resultado = chain.invoke({"input": texto})

# resultado é uma instância de EventsList — acesso direto aos campos
print(f"Eventos extraídos: {len(resultado.events)}\n")
for ev in resultado.events:
    print(f"  [{ev.date}] {ev.event}")

In [ ]:
# O objeto Pydantic também pode ser serializado para dict ou JSON facilmente
import json

print(json.dumps(resultado.model_dump(), indent=2, ensure_ascii=False))

## 4. Extração a partir de páginas web

A mesma abordagem funciona com qualquer fonte de texto — incluindo páginas HTML carregadas
com o `WebBaseLoader`.

O loader extrai o texto visível da página e o passa para a chain de extração.

> **Atenção ao tamanho**: páginas web podem ser muito longas. Se o conteúdo ultrapassar
> o limite de contexto do modelo, considere fazer o split do texto antes de extrair.

In [ ]:
from langchain_community.document_loaders import WebBaseLoader

# Carregando o conteúdo da página
loader = WebBaseLoader("https://www.techtudo.com.br")
page = loader.load()

print(f"Páginas carregadas: {len(page)}")
print(f"Tamanho do conteúdo: {len(page[0].page_content)} caracteres")
print(f"\nTrecho inicial:\n{page[0].page_content[:300]}")

## 5. Schema para extração de posts de blog

Definimos um schema para extrair postagens de uma página de blog.
O campo `author` pode ser opcional — nem toda página exibe o autor diretamente.

Usamos `Optional[str]` para indicar que o campo pode estar ausente no texto.

> **`Optional[str]`** diz ao modelo que, se não encontrar a informação, pode retornar `None`
> em vez de inventar um valor. Isso é importante para extração confiável.

In [ ]:
from typing import Optional

class BlogPost(BaseModel):
    """Detalhes sobre uma postagem de blog."""
    title:  str           = Field(description="Título da postagem no blog")
    author: Optional[str] = Field(
        default=None,
        description="Nome do autor da postagem. None se não encontrado."
    )

class BlogSite(BaseModel):
    """Conjunto de postagens extraídas de um site de blog."""
    posts: List[BlogPost] = Field(
        description="Coleção de postagens encontradas na página"
    )

In [ ]:
prompt_blog = ChatPromptTemplate.from_messages([
    ("system", "Extraia da página os posts do blog com as informações especificadas."),
    ("human", "{input}")
])

# Substitui o antigo JsonKeyOutputFunctionsParser(key_name='posts')
# Agora acessamos resultado.posts diretamente
chain_blog = prompt_blog | chat.with_structured_output(BlogSite)

resultado_blog = chain_blog.invoke({"input": page})

In [ ]:
# Acessando a lista de posts diretamente via atributo
# Antes era necessário JsonKeyOutputFunctionsParser(key_name='posts') para isso
print(f"Posts extraídos: {len(resultado_blog.posts)}\n")
for i, post in enumerate(resultado_blog.posts, 1):
    autor = post.author or "(não informado)"
    print(f"[{i:>2}] {post.title}")
    print(f"      Autor: {autor}")

## 6. Comparando os dois parsers antigos com a abordagem moderna

No código original havia dois parsers diferentes dependendo do que se queria acessar.
Com `with_structured_output()`, tudo é simplificado:

| Objetivo | Antigo | Moderno |
|----------|--------|---------|
| Objeto inteiro como dict | `JsonOutputFunctionsParser()` | `resultado.model_dump()` |
| Apenas uma chave do dict | `JsonKeyOutputFunctionsParser(key_name='posts')` | `resultado.posts` |
| Objeto Pydantic tipado | não havia | `resultado` (já é Pydantic) |

A abordagem moderna é superior porque:
- **Type safety**: seu editor sabe os campos disponíveis e suas tipagens
- **Validação automática**: o Pydantic valida cada campo na hora da extração
- **Menos boilerplate**: uma linha substitui três componentes encadeados

In [ ]:
# Demonstrando os diferentes formatos de saída disponíveis
resultado_eventos = chain.invoke({"input": texto})

# 1. Objeto Pydantic — acesso por atributo
print("1. Objeto Pydantic:")
print(resultado_eventos.events[0])
print()

# 2. Dict completo — equivalente ao JsonOutputFunctionsParser()
print("2. Dict completo (model_dump()):")
print(resultado_eventos.model_dump())
print()

# 3. Apenas a lista — equivalente ao JsonKeyOutputFunctionsParser(key_name='events')
print("3. Apenas a lista de eventos:")
print(resultado_eventos.events)

## Resumo

| Conceito | Descrição |
|----------|-----------|
| Schema aninhado | Um `BaseModel` com `List[OutroModel]` extrai múltiplos objetos |
| `Optional[str]` | Permite que o modelo retorne `None` quando o dado não existe |
| `with_structured_output(Schema)` | Substitui `bind(functions)` + todos os parsers |
| `.model_dump()` | Converte o objeto Pydantic para dict (antes era `JsonOutputFunctionsParser`) |
| `WebBaseLoader` | Carrega texto de páginas web para usar como input da chain |

**Padrão moderno completo para extração:**

```python
from pydantic import BaseModel, Field
from typing import List, Optional

class Item(BaseModel):
    """Descrição do item a extrair."""
    campo: str           = Field(description="O que extrair")
    opcional: Optional[str] = Field(default=None, description="Campo que pode não existir")

class Colecao(BaseModel):
    """Coleção de itens extraídos."""
    items: List[Item] = Field(description="Todos os itens encontrados")

chain = prompt | chat.with_structured_output(Colecao)
resultado = chain.invoke({"input": texto})

# Acesso direto, tipado, sem parsers extras
for item in resultado.items:
    print(item.campo)
```

> **Próximo passo**: usar extração estruturada em conjunto com **agentes** para automatizar
> o processamento de documentos em fluxos mais complexos.